# 2.1 数据获取

## 学习目标
- 使用 yfinance 获取美股/全球数据
- 使用 AKShare 获取 A 股数据
- 了解数据字段含义（OHLCV）
- 将数据保存为 CSV 供后续使用

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from pathlib import Path

# 数据存储路径
DATA_DIR = Path('../datasets')
DATA_DIR.mkdir(exist_ok=True)
print('准备就绪 ✅')

## 1. yfinance — 美股数据

In [ ]:
# ===== 单只股票 =====
ticker = yf.Ticker('AAPL')

# 日线历史数据
hist = ticker.history(start='2020-01-01', end='2024-01-01')
print('AAPL 历史数据字段:', hist.columns.tolist())
print(f'数据量: {len(hist)} 行')
hist.tail(3)

In [ ]:
# ===== 字段说明 =====
field_desc = {
    'Open':  '开盘价 — 当日第一笔成交价',
    'High':  '最高价 — 当日最高成交价',
    'Low':   '最低价 — 当日最低成交价',
    'Close': '收盘价 — 当日最后成交价（原始）',
    'Volume':'成交量 — 当日总成交股数',
    'Dividends': '分红 — 当日每股分红金额',
    'Stock Splits': '股票拆分倍数（2=一股变两股）'
}
for k, v in field_desc.items():
    if k in hist.columns:
        print(f'  {k:<15}: {v}')

In [ ]:
# ===== 多只股票批量下载 =====
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'SPY']
prices = yf.download(tickers, start='2020-01-01', end='2024-01-01',
                     progress=False)['Close']

# 标准化为基准100（便于比较）
normalized = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 5))
for col in normalized.columns:
    ax.plot(normalized.index, normalized[col], label=col, linewidth=1.3)
ax.axhline(100, color='gray', linestyle='--', alpha=0.5)
ax.set_title('科技股 vs SPY 标准化收益对比 (基准=100)', fontsize=13)
ax.set_ylabel('标准化价格')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 保存到 datasets
prices.to_csv(DATA_DIR / 'sp500_sample.csv')
print(f'已保存至 {DATA_DIR / "sp500_sample.csv"}')

## 2. AKShare — A 股数据

In [ ]:
try:
    import akshare as ak

    # 获取贵州茅台（600519）日线数据
    mao_tai = ak.stock_zh_a_hist(
        symbol='600519',
        period='daily',
        start_date='20220101',
        end_date='20240101',
        adjust='qfq'  # 前复权
    )
    print('贵州茅台数据字段:', mao_tai.columns.tolist())
    mao_tai = mao_tai.rename(columns={
        '日期': 'Date', '开盘': 'Open', '最高': 'High',
        '最低': 'Low', '收盘': 'Close', '成交量': 'Volume'
    })
    mao_tai['Date'] = pd.to_datetime(mao_tai['Date'])
    mao_tai = mao_tai.set_index('Date')

    # 保存
    mao_tai[['Open', 'High', 'Low', 'Close', 'Volume']].to_csv(DATA_DIR / 'stock_daily.csv')
    print(f'已保存 {len(mao_tai)} 条A股数据')
    mao_tai.tail(3)

except ImportError:
    print('⚠️  akshare 未安装，跳过A股示例')
    print('   安装命令: pip install akshare')
    # 改用示例CSV（datasets/stock_daily.csv）
    df_sample = pd.read_csv(DATA_DIR / 'stock_daily.csv', index_col=0, parse_dates=True)
    print('已加载示例A股数据:')
    print(df_sample.tail(3))

## 3. 读取本地 CSV 数据

In [ ]:
# 读取已保存的数据
sp500 = pd.read_csv(DATA_DIR / 'sp500_sample.csv', index_col=0, parse_dates=True)
print('读取成功！')
print(f'形状: {sp500.shape}')
sp500.tail(3)

## 🎯 练习

1. 下载你感兴趣的 5 只股票，绘制标准化收益对比图。
2. 使用 `yf.Ticker('AAPL').info` 查看公司基本面信息（PE、市值等）。
3. 将数据保存为 CSV，明天重新读取，不再需要重新下载。

---
**下一节** → `02_data_cleaning.ipynb`